# Data loader smoke test (M1-T05)

Proves the data loader works on the real BraTS 2021 data: it cleans a few training and validation patients with
Yasmin's pipeline (`BrainMRI_Data/pipeline/standardize.py`, M1-T04), then trains a tiny 3D U-Net for **2 epochs**.
**Done when:** the last cell prints `SMOKE TEST PASSED`.

**How to run:** **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**. About 15–25 minutes,
most of it unpacking BraTS 2021 (13 GB) onto the Colab disk.

Only patients from the `train` and `val` groups of `qc/split_v1.csv` are used; the loader refuses locked test patients.

In [ ]:
# 1. Settings and Drive
CASES, VAL_CASES = 16, 4          # training / validation patients for the smoke test
import os, sys, json
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")
BASE = Path("/content/drive/MyDrive/BrainMRI_Data")
SPLIT_CSV = BASE / "qc/split_v1.csv"
PROCESSED = Path("/content/processed")
OUT = BASE / "runs/M1-T05_smoke"
OUT.mkdir(parents=True, exist_ok=True)
assert SPLIT_CSV.exists(), f"missing {SPLIT_CSV}: the split from M1-T04 is needed"
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "No GPU: the test still runs on CPU, just slower"

In [ ]:
# 2. Install MONAI (PyTorch, nibabel, pandas and PyYAML are already on Colab)
!pip -q install "monai>=1.4"

In [ ]:
# 3. The loader code (a copy of src/bmts, configs/data/loader_v1.yaml and scripts/smoke_train.py from the repo).
#    Once the repo exists, you can `git clone` it to /content/repo instead and skip this cell.
CODE = Path("/content/repo")
FILES = {
 "src/bmts/__init__.py": "\"\"\"Brain MRI tumor system (bmts): DEPI graduation project package.\"\"\"\n",
 "src/bmts/common/__init__.py": "",
 "src/bmts/common/constants.py": "\"\"\"Project-wide constants: MRI modalities and the canonical label scheme.\n\nCanonical labels (docs/PROJECT_STRUCTURE.md \u00a73.2): 0 background, 1 NCR/NET, 2 ED, 3 ET, 4 RC (post-treatment only).\nThe cleaning step (M1-T04) writes every mask in this scheme, e.g. BraTS 2021's ET label 4 becomes 3.\n\"\"\"\n\nMODALITIES = (\"t1\", \"t1ce\", \"t2\", \"flair\")  # channel order of every model input\nSEG = \"seg\"\n\nBACKGROUND, NCR, ED, ET, RC = 0, 1, 2, 3, 4\n\n# Evaluation regions, in the channel order of the model output. RC is never part of a tumor region.\nREGIONS = {\n    \"TC\": (NCR, ET),       # tumor core\n    \"WT\": (NCR, ED, ET),   # whole tumor\n    \"ET\": (ET,),           # enhancing tumor\n}\n",
 "src/bmts/data/__init__.py": "",
 "src/bmts/data/augmentation/__init__.py": "",
 "src/bmts/data/augmentation/transforms.py": "\"\"\"MONAI transform chains for training and validation (M1-T05).\n\nInput: one cleaned case = {t1, t1ce, t2, flair, seg} file paths (output of M1-T04, already RAS,\n240x240x155, z-scored per modality, masks in the canonical label scheme).\nOutput: \"image\" (4, H, W, D) float32 in MODALITIES order, \"label\" (3, H, W, D) float32 region masks (TC, WT, ET).\n\"\"\"\nfrom __future__ import annotations\n\nimport math\n\nimport torch\nfrom monai import transforms as T\nfrom monai.data import MetaTensor\n\nfrom bmts.common.constants import MODALITIES, REGIONS, SEG\n\nFOREGROUND = \"fg\"  # temporary whole-tumor mask used only to pick training patches\n\n\ndef nonzero(x):\n    \"\"\"Brain voxels: the cleaned scans are z-scored, so the brain has negative values too; background is exactly 0.\"\"\"\n    return x != 0\n\n\nclass SegToRegionsd(T.MapTransform):\n    \"\"\"Canonical label map (1 channel) -> one binary channel per region in REGIONS (TC, WT, ET).\n\n    Also writes a 1-channel whole-tumor mask under `foreground_key` for tumor-centred patch sampling.\n    Labels outside every region (background, resection cavity 4) end up 0 in every channel.\n    \"\"\"\n\n    def __init__(self, keys=SEG, label_key=\"label\", foreground_key=FOREGROUND, regions=REGIONS):\n        super().__init__(keys)\n        self.label_key, self.foreground_key = label_key, foreground_key\n        self.regions = {name: torch.tensor(labels) for name, labels in regions.items()}\n\n    def __call__(self, data):\n        d = dict(data)\n        for key in self.key_iterator(d):\n            seg = torch.as_tensor(d[key])[0].round().long()\n            channels = torch.stack([torch.isin(seg, labels) for labels in self.regions.values()]).float()\n            whole = channels[list(self.regions).index(\"WT\")][None] if \"WT\" in self.regions else channels.amax(0, keepdim=True)\n            if isinstance(d[key], MetaTensor):\n                channels = MetaTensor(channels, meta=d[key].meta)\n                whole = MetaTensor(whole, meta=d[key].meta)\n            d[self.label_key], d[self.foreground_key] = channels, whole\n            del d[key]\n        return d\n\n\ndef load_transforms(normalize=False):\n    \"\"\"Read the 5 files of a case, stack the 4 MRI types as channels, turn the mask into region channels.\"\"\"\n    keys = [*MODALITIES, SEG]\n    chain = [\n        T.LoadImaged(keys, image_only=True),\n        T.EnsureChannelFirstd(keys),\n        T.ConcatItemsd(list(MODALITIES), \"image\"),\n        T.DeleteItemsd(list(MODALITIES)),\n        T.Orientationd([\"image\", SEG], axcodes=\"RAS\", labels=((\"L\", \"R\"), (\"P\", \"A\"), (\"I\", \"S\"))),\n        T.EnsureTyped(\"image\", dtype=torch.float32),\n    ]\n    if normalize:  # only for scans that were not z-scored by the cleaning step\n        chain.append(T.NormalizeIntensityd(\"image\", nonzero=True, channel_wise=True))\n    chain.append(SegToRegionsd(SEG))\n    return chain\n\n\ndef train_transforms(cfg):\n    \"\"\"Loading + random tumor-centred patch + augmentation (flips, small rotation/scaling, intensity, gamma).\"\"\"\n    roi = cfg[\"roi_size\"]\n    aug, sampling = cfg[\"augment\"], cfg[\"sampling\"]\n    keys = [\"image\", \"label\", FOREGROUND]\n    chain = load_transforms(cfg.get(\"normalize\", False)) + [\n        T.CropForegroundd(keys, source_key=\"image\", select_fn=nonzero, allow_smaller=True),\n        T.SpatialPadd(keys, spatial_size=roi),\n        T.RandCropByPosNegLabeld([\"image\", \"label\"], label_key=FOREGROUND, spatial_size=roi,\n                                 pos=sampling[\"pos\"], neg=sampling[\"neg\"], num_samples=1),\n        T.DeleteItemsd(FOREGROUND),\n    ]\n    for axis in range(3):\n        chain.append(T.RandFlipd([\"image\", \"label\"], prob=aug[\"flip_prob\"], spatial_axis=axis))\n    if aug[\"affine_prob\"] > 0:\n        angle = math.radians(aug[\"rotate_deg\"])\n        chain.append(T.RandAffined([\"image\", \"label\"], prob=aug[\"affine_prob\"], rotate_range=(angle,) * 3,\n                                   scale_range=(aug[\"scale\"],) * 3, mode=(\"bilinear\", \"nearest\"),\n                                   padding_mode=\"zeros\"))\n    if aug.get(\"elastic_prob\", 0) > 0:\n        chain.append(T.Rand3DElasticd([\"image\", \"label\"], sigma_range=(5, 8), magnitude_range=(100, 200),\n                                      prob=aug[\"elastic_prob\"], mode=(\"bilinear\", \"nearest\")))\n    chain += [\n        T.RandScaleIntensityd(\"image\", factors=0.1, prob=aug[\"intensity_prob\"]),\n        T.RandShiftIntensityd(\"image\", offsets=0.1, prob=aug[\"intensity_prob\"]),\n        T.RandAdjustContrastd(\"image\", gamma=(0.7, 1.5), prob=aug[\"gamma_prob\"]),\n        T.EnsureTyped([\"image\", \"label\"], dtype=torch.float32, track_meta=False),\n    ]\n    return T.Compose(chain)\n\n\ndef val_transforms(cfg):\n    \"\"\"Loading only: the whole 240x240x155 volume, for sliding-window evaluation.\"\"\"\n    return T.Compose(load_transforms(cfg.get(\"normalize\", False)) + [\n        T.DeleteItemsd(FOREGROUND),\n        T.EnsureTyped([\"image\", \"label\"], dtype=torch.float32, track_meta=False),\n    ])\n",
 "src/bmts/data/datasets/__init__.py": "",
 "src/bmts/data/datasets/brats.py": "\"\"\"Find the cleaned cases of a split and build the training / validation data loaders (M1-T05).\n\nCleaned layout (M1-T04, Yasmin's pipeline/standardize.py):\n    <processed_root>/<dataset>/<subject>/<timepoint>/{t1,t1ce,t2,flair,seg}.nii.gz\nSplit file (M1-T04, qc/split_v1.csv): one row per patient with dataset_source, canonical_subject_id, split, lockbox.\n\nLocked patients (split \"test\" or lockbox True) are refused unless allow_lockbox=True, which only the frozen\nfinal evaluation (M5-T02) may set.\n\"\"\"\nfrom __future__ import annotations\n\nimport warnings\nfrom pathlib import Path\n\nimport pandas as pd\nimport torch\nimport yaml\nfrom monai.data import CacheDataset, DataLoader, Dataset\n\nfrom bmts.common.constants import MODALITIES, SEG\nfrom bmts.data.augmentation.transforms import train_transforms, val_transforms\n\nREQUIRED_COLUMNS = {\"dataset_source\", \"canonical_subject_id\", \"split\"}\n\n\nclass LockboxError(RuntimeError):\n    \"\"\"Raised when code asks for locked test patients outside the final evaluation.\"\"\"\n\n\ndef load_config(path, **overrides):\n    \"\"\"Read a loader YAML config; keyword overrides replace top-level keys (None values are ignored).\"\"\"\n    cfg = yaml.safe_load(Path(path).read_text(encoding=\"utf-8\"))\n    cfg.update({k: v for k, v in overrides.items() if v is not None})\n    return cfg\n\n\ndef list_cases(split_csv, processed_root, splits=(\"train\",), datasets=(\"brats2021\",), limit=None,\n               allow_lockbox=False, on_missing=\"error\"):\n    \"\"\"Return one dict per (patient, timepoint): {\"id\", \"t1\", \"t1ce\", \"t2\", \"flair\", \"seg\"} file paths.\n\n    limit keeps the first N patients (sorted by ID) before the files are checked, for quick smoke tests.\n    on_missing: \"error\" stops on the first incomplete case list; \"skip\" drops those cases with a warning.\n    \"\"\"\n    table = pd.read_csv(split_csv, dtype=str, keep_default_na=False)\n    missing_columns = REQUIRED_COLUMNS - set(table.columns)\n    if missing_columns:\n        raise ValueError(f\"{split_csv} lacks columns {sorted(missing_columns)}\")\n    rows = table[table.split.isin(splits) & table.dataset_source.isin(datasets)]\n    locked = rows.split.eq(\"test\")\n    if \"lockbox\" in rows:\n        locked |= rows.lockbox.str.lower().isin([\"true\", \"1\", \"yes\"])\n    if locked.any() and not allow_lockbox:\n        raise LockboxError(f\"{int(locked.sum())} requested patients are in the locked test set; \"\n                           \"they may only be loaded by the final evaluation (M5-T02).\")\n    rows = rows.sort_values(\"canonical_subject_id\")\n    if limit is not None:\n        rows = rows.head(limit)\n\n    cases, problems = [], []\n    for row in rows.itertuples():\n        subject = row.canonical_subject_id.split(\"__\", 1)[-1]\n        subject_dir = Path(processed_root) / row.dataset_source / subject\n        timepoints = sorted(p for p in subject_dir.iterdir() if p.is_dir()) if subject_dir.is_dir() else []\n        if not timepoints:\n            problems.append(f\"{row.canonical_subject_id}: no cleaned scans in {subject_dir}\")\n            continue\n        for tp in timepoints:\n            files = {k: tp / f\"{k}.nii.gz\" for k in (*MODALITIES, SEG)}\n            absent = [k for k, p in files.items() if not p.is_file()]\n            if absent:\n                problems.append(f\"{row.canonical_subject_id}/{tp.name}: missing {', '.join(absent)}\")\n                continue\n            cases.append({\"id\": f\"{row.canonical_subject_id}/{tp.name}\", **{k: str(p) for k, p in files.items()}})\n\n    if problems:\n        message = f\"{len(problems)} of {len(rows)} patients are incomplete, e.g.\\n  \" + \"\\n  \".join(problems[:5])\n        if on_missing == \"error\":\n            raise FileNotFoundError(message + \"\\nRun the cleaning step for them, or set on_missing: skip.\")\n        warnings.warn(message, stacklevel=2)\n    return cases\n\n\ndef build_loaders(cfg, train_limit=None, val_limit=None):\n    \"\"\"Training loader (random augmented patches) and validation loader (whole volumes, batch size 1).\"\"\"\n    common = dict(split_csv=cfg[\"split_csv\"], processed_root=cfg[\"processed_root\"], datasets=cfg[\"datasets\"],\n                  on_missing=cfg.get(\"on_missing\", \"error\"))\n    train_cases = list_cases(splits=(cfg[\"train_split\"],), limit=train_limit, **common)\n    val_cases = list_cases(splits=(cfg[\"val_split\"],), limit=val_limit, **common)\n    if not train_cases:\n        raise RuntimeError(\"No training cases found: check split_csv, processed_root and datasets in the config.\")\n\n    cache_rate = cfg.get(\"cache_rate\", 0.0)\n    workers = cfg.get(\"num_workers\", 0)\n    if cache_rate > 0:\n        train_ds = CacheDataset(train_cases, train_transforms(cfg), cache_rate=cache_rate, num_workers=workers)\n    else:\n        train_ds = Dataset(train_cases, train_transforms(cfg))\n    val_ds = Dataset(val_cases, val_transforms(cfg))\n    extra = dict(num_workers=workers, pin_memory=torch.cuda.is_available(), persistent_workers=workers > 0)\n    train_loader = DataLoader(train_ds, batch_size=cfg[\"batch_size\"], shuffle=True, drop_last=False, **extra)\n    val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, **extra)\n    return train_loader, val_loader\n",
 "configs/data/loader_v1.yaml": "# Data loader settings (M1-T05). Read by bmts.data.datasets.brats.load_config / build_loaders.\n# Paths are for Colab; override them on the command line elsewhere (scripts/smoke_train.py --help).\n\nprocessed_root: /content/processed                          # cleaned scans: <dataset>/<subject>/<timepoint>/{t1,t1ce,t2,flair,seg}.nii.gz\nsplit_csv: /content/drive/MyDrive/BrainMRI_Data/qc/split_v1.csv   # patient-level split from M1-T04\ndatasets: [brats2021]        # the main model trains on BraTS 2021 only (docs/dataset_research.md \u00a78.2, rule 4)\ntrain_split: train\nval_split: val\non_missing: error            # error | skip (skip drops cases whose cleaned files are missing, with a warning)\nnormalize: false             # cleaned scans are already z-scored per MRI type; true only for uncleaned scans\n\nroi_size: [128, 128, 128]    # training patch in voxels (1 mm); use [96, 96, 96] on GPUs with less than 8 GB\nbatch_size: 2\nnum_workers: 2               # Colab has 2 CPU cores\ncache_rate: 0.0              # share of training cases kept in RAM; one case is about 150 MB, Colab has about 12 GB\n\nsampling:                    # training patches centred on tumor (pos) versus on other brain tissue (neg)\n  pos: 2\n  neg: 1\n\naugment:                     # proposal Milestone 1: rotations, flips, scaling, intensity/gamma shifts, elastic\n  flip_prob: 0.5             # per axis\n  affine_prob: 0.2           # small rotation + scaling\n  rotate_deg: 15\n  scale: 0.1\n  intensity_prob: 0.5        # brightness scale and shift\n  gamma_prob: 0.15\n  elastic_prob: 0.0          # elastic deformation is slow on CPU; switch on in a later experiment if useful\n\nseed: 42\n",
 "scripts/smoke_train.py": "\"\"\"M1-T05 smoke test: prove the data loader feeds a model without errors (2 tiny epochs).\n\n    python scripts/smoke_train.py --config configs/data/loader_v1.yaml --cases 8 --epochs 2\n\nIt trains a deliberately small 3D U-Net on a few training patients, runs one validation pass on a few\nvalidation patients, and writes a JSON report. The scores mean nothing after 2 epochs: the point is that\nevery batch loads with the right shapes and values. Locked test patients are never read.\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport sys\nimport time\nfrom pathlib import Path\n\nsys.path.insert(0, str(Path(__file__).resolve().parents[1] / \"src\"))\n\nimport torch  # noqa: E402\nfrom monai.inferers import sliding_window_inference  # noqa: E402\nfrom monai.losses import DiceLoss  # noqa: E402\nfrom monai.metrics import DiceMetric  # noqa: E402\nfrom monai.networks.nets import UNet  # noqa: E402\nfrom monai.utils import set_determinism  # noqa: E402\n\nfrom bmts.common.constants import MODALITIES, REGIONS  # noqa: E402\nfrom bmts.data.datasets.brats import build_loaders, load_config  # noqa: E402\n\n\ndef check_batch(batch, roi):\n    \"\"\"Fail loudly if a training batch does not have the shapes and values the model expects.\"\"\"\n    image, label = batch[\"image\"], batch[\"label\"]\n    assert image.ndim == 5 and image.shape[1] == len(MODALITIES), f\"image shape {tuple(image.shape)}\"\n    assert label.shape[1] == len(REGIONS) and label.shape[2:] == image.shape[2:], f\"label shape {tuple(label.shape)}\"\n    assert list(image.shape[2:]) == list(roi), f\"patch {tuple(image.shape[2:])} != roi {roi}\"\n    assert torch.isfinite(image).all(), \"image has NaN/Inf\"\n    assert set(label.unique().tolist()) <= {0.0, 1.0}, \"label is not binary\"\n\n\ndef main():\n    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)\n    p.add_argument(\"--config\", default=\"configs/data/loader_v1.yaml\")\n    p.add_argument(\"--processed-root\", help=\"override processed_root\")\n    p.add_argument(\"--split-csv\", help=\"override split_csv\")\n    p.add_argument(\"--cases\", type=int, default=8, help=\"training patients to use\")\n    p.add_argument(\"--val-cases\", type=int, default=2, help=\"validation patients to use\")\n    p.add_argument(\"--epochs\", type=int, default=2)\n    p.add_argument(\"--roi\", type=int, nargs=3, help=\"patch size, e.g. 96 96 96\")\n    p.add_argument(\"--batch-size\", type=int)\n    p.add_argument(\"--num-workers\", type=int)\n    p.add_argument(\"--report\", default=\"smoke_report.json\")\n    args = p.parse_args()\n\n    cfg = load_config(args.config, processed_root=args.processed_root, split_csv=args.split_csv,\n                      roi_size=args.roi, batch_size=args.batch_size, num_workers=args.num_workers)\n    set_determinism(cfg.get(\"seed\", 42))\n    device = torch.device(\"cuda\" if torch.cuda.is_available() else \"cpu\")\n    amp = device.type == \"cuda\"\n    print(f\"device: {device}{' (' + torch.cuda.get_device_name(0) + ')' if amp else ''}\")\n\n    train_loader, val_loader = build_loaders(cfg, train_limit=args.cases, val_limit=args.val_cases)\n    print(f\"training cases: {len(train_loader.dataset)}, validation cases: {len(val_loader.dataset)}, \"\n          f\"patch {cfg['roi_size']}, batch {cfg['batch_size']}\")\n\n    model = UNet(spatial_dims=3, in_channels=len(MODALITIES), out_channels=len(REGIONS),\n                 channels=(16, 32, 64, 128), strides=(2, 2, 2), num_res_units=1).to(device)\n    loss_fn = DiceLoss(sigmoid=True, smooth_nr=1e-5, smooth_dr=1e-5)\n    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)\n    scaler = torch.amp.GradScaler(\"cuda\", enabled=amp)\n    report = {\"device\": str(device), \"config\": cfg, \"epochs\": []}\n\n    for epoch in range(1, args.epochs + 1):\n        model.train()\n        start, losses = time.time(), []\n        for step, batch in enumerate(train_loader):\n            check_batch(batch, cfg[\"roi_size\"])\n            if epoch == 1 and step == 0:\n                img, lab = batch[\"image\"], batch[\"label\"]\n                print(f\"first batch: image {tuple(img.shape)} {img.dtype}, label {tuple(lab.shape)}, \"\n                      f\"tumor voxels per region {[int(v) for v in lab.sum(dim=(0, 2, 3, 4))]}\")\n            image, label = batch[\"image\"].to(device), batch[\"label\"].to(device)\n            optimizer.zero_grad(set_to_none=True)\n            with torch.autocast(device.type, enabled=amp):\n                loss = loss_fn(model(image), label)\n            assert torch.isfinite(loss), f\"loss is {loss.item()}\"\n            scaler.scale(loss).backward()\n            scaler.step(optimizer)\n            scaler.update()\n            losses.append(loss.item())\n        train_time = time.time() - start\n\n        model.eval()\n        dice = DiceMetric(include_background=True, reduction=\"mean_batch\")\n        start = time.time()\n        with torch.no_grad(), torch.autocast(device.type, enabled=amp):\n            for batch in val_loader:\n                image, label = batch[\"image\"].to(device), batch[\"label\"].to(device)\n                logits = sliding_window_inference(image, cfg[\"roi_size\"], 1, model, overlap=0.25)\n                dice((torch.sigmoid(logits.float()) > 0.5).float(), label)\n        scores = dice.aggregate().tolist() if len(val_loader.dataset) else []\n        entry = {\"epoch\": epoch, \"train_batches\": len(losses), \"mean_loss\": sum(losses) / len(losses),\n                 \"train_seconds\": round(train_time, 1), \"val_seconds\": round(time.time() - start, 1),\n                 \"val_dice\": dict(zip(REGIONS, [round(s, 4) for s in scores]))}\n        report[\"epochs\"].append(entry)\n        print(f\"epoch {epoch}: {entry['train_batches']} batches, loss {entry['mean_loss']:.4f}, \"\n              f\"{entry['train_seconds']}s train, {entry['val_seconds']}s val, val Dice {entry['val_dice']}\")\n\n    if amp:\n        report[\"peak_gpu_memory_gb\"] = round(torch.cuda.max_memory_allocated() / 1e9, 2)\n        print(f\"peak GPU memory: {report['peak_gpu_memory_gb']} GB\")\n    report[\"result\"] = \"passed\"\n    Path(args.report).write_text(json.dumps(report, indent=2, default=str), encoding=\"utf-8\")\n    print(f\"SMOKE TEST PASSED: every batch loaded with the right shapes and values. Report: {args.report}\")\n\n\nif __name__ == \"__main__\":\n    main()\n"
}
for rel, text in FILES.items():
    path = CODE / rel
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")
print("wrote", len(FILES), "files to", CODE)

In [ ]:
# 4. Clean the chosen patients with Yasmin's pipeline (unpacks BraTS 2021 on the Colab disk first)
import pandas as pd
sys.path.insert(0, str(BASE / "pipeline"))
import standardize as S

split = pd.read_csv(SPLIT_CSV, dtype=str)
split = split[split.dataset_source == "brats2021"].sort_values("canonical_subject_id")
def first(group, n):   # the same patients the loader picks with --cases / --val-cases
    return split[split.split == group].canonical_subject_id.str.split("__").str[1].head(n).tolist()
subjects = first("train", CASES) + first("val", VAL_CASES)
print(f"{len(subjects)} patients: {subjects[:3]} ...")

S.ensure_extracted("brats2021")
jobs = S.run(["brats2021"], subjects=set(subjects))
assert (jobs.status != "failed").all(), jobs[jobs.status == "failed"][["subject", "scan", "error"]]

In [ ]:
# 5. The smoke test: 2 tiny epochs on the chosen patients
%cd /content/repo
!python scripts/smoke_train.py --config configs/data/loader_v1.yaml --processed-root /content/processed --split-csv "{SPLIT_CSV}" --cases {CASES} --val-cases {VAL_CASES} --epochs 2 --report "{OUT}/smoke_report.json"

In [ ]:
# 6. Result (also saved to MyDrive/BrainMRI_Data/runs/M1-T05_smoke/smoke_report.json)
report = json.loads((OUT / "smoke_report.json").read_text())
print("result:", report["result"], "| device:", report["device"], "| peak GPU memory (GB):", report.get("peak_gpu_memory_gb"))
for e in report["epochs"]:
    print(e)
drive.flush_and_unmount()

**Next:** the smoke report shows how long an epoch takes and how much GPU memory a 128³ patch needs. M2-T03
(baseline training) uses the same `build_loaders` with more patients and epochs. The Dice scores after 2 epochs
mean nothing; they only show that the validation path runs.